[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P60-Tier-7_executed.ipynb)

---


# 60. The Warehouse Slotting Optimization Problem

## Tier 7: The Human–AI Symbiotic Partnership

### Purpose and scope

This notebook demonstrates a human-in-the-loop workflow for warehouse slotting. A random-forest model is trained on synthetic slotting scenarios, and three transparent, weighted greedy policies generate candidate assignments. A simulated human-review policy then accepts, modifies, or rejects each recommendation and records changes in expert trust scores.

The code is an instructional simulation, not a validated warehouse optimization system. In particular, the human decisions are generated by explicit rules rather than collected from people, and the recorded trust changes do not retrain the model or alter subsequent recommendations. The numerical results produced by running the notebook depend on the generated synthetic data; they are not operational performance claims.

### Modeling assumptions

- Each SKU is assigned to at most one location. A location can contain multiple SKUs as long as its space and weight limits are respected.
- Assignment feasibility is determined only by space and weight. Other real-world rules listed in the code, such as cold-chain requirements, security separation, labor availability, and relocation scheduling, are not enforced by the assignment procedure.
- Location accessibility and zone scores are simplified proxies for operational suitability. Travel paths, order co-occurrence, replenishment effort, and actual picking times are not modeled.
- Synthetic training scenarios are generated by random feasible assignments. Their efficiency scores are constructed from the same simplified score used to evaluate recommendations; they are not historical warehouse outcomes.
- The model and human-review rules illustrate components of a collaborative workflow. They do not establish that the system is explainable, calibrated, or suitable for autonomous decisions.

### Method and mathematical representation

A **greedy assignment** makes one placement at a time: it considers SKUs in descending velocity order and assigns each SKU to the feasible location with the highest current weighted score. This is computationally transparent, but it is not guaranteed to find a globally optimal assignment. The notebook creates three candidates by changing the weights in that score; it does not solve a mathematical program or prove optimality.

For SKU \(i\) and location \(j\), the placement score used by a scenario is

$$
S_{ij} = w_v \left(\frac{v_i}{v_{\max}}\right)\left(\frac{10-a_j}{9}\right)
        + w_z q_j
        + w_u r_j,
\qquad w_v+w_z+w_u=1.
$$

Here, \(v_i\) is SKU velocity, \(v_{\max}\) is the largest velocity in the instance, \(a_j\) is location accessibility (smaller values represent easier access), \(q_j\) is the location's zone score, and \(r_j\) is a utilization contribution based on the location's current space use. The weights \(w_v,w_z,w_u\) express the scenario's priorities. The code's `zone_scores` and `util_score` implement \(q_j\) and \(r_j\); `velocity_score` implements the normalized velocity-accessibility term.

A placement is feasible when adding SKU \(i\) keeps both resource totals within capacity:

$$
U_j^{\text{space}} + s_i \le C_j,
\qquad
U_j^{\text{weight}} + m_i \le W_j.
$$

In this expression, \(s_i\) and \(m_i\) are the SKU's space requirement and weight; \(C_j\) and \(W_j\) are the location's space capacity and weight limit; and \(U_j^{\text{space}}\) and \(U_j^{\text{weight}}\) are the totals already assigned there. In the implementation these quantities are stored in `sku`, `location`, and `location_usage` dictionaries.

The assignment's illustrative efficiency score is the mean, over assigned SKUs, of a weighted velocity-accessibility term, a zone preference, and a utilization bonus or penalty. This is a scoring rule, not a measured picking-efficiency percentage. In the code it is computed by `_calculate_efficiency_score` and is used to compare candidates consistently with the synthetic training targets.

The random forest estimates this score from ten assignment-level features. Its role is to provide a learned estimate and a held-out error diagnostic; the transparent greedy policies, rather than the model's predictions, construct the candidate assignments. Human review follows those recommendations and records the simulated decision and trust update.

For \(N\) SKUs, \(L\) locations, and \(K\) generated scenarios, the straightforward assignment-generation work is approximately \(O(KNL)\), plus model training and prediction costs. This example is intended to make the workflow inspectable, not to claim a scalable or globally optimal solution.

### Workflow

1. Generate synthetic SKUs, locations, and historical-style scenarios.
2. Train and evaluate a random-forest regressor on the synthetic scenarios.
3. Generate three weighted greedy slotting recommendations.
4. Produce rule-based explanations and heuristic confidence indicators.
5. Apply a simulated human-review policy and record decisions and trust scores.
6. Summarize repeated review cycles and visualize the recorded metrics.

### How to interpret the results

Treat printed scores, model error, acceptance rates, and trust metrics as outputs of the stated synthetic simulation only. A high acceptance rate is not evidence of recommendation quality, and a trust-score increase is not evidence that the model learned. The dashboard reports the simulated process; operational claims would require representative data, validated outcome measures, and review by warehouse experts.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
import random
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

np.random.seed(42)
random.seed(42)


@dataclass
class HumanAISystem:
    """Synthetic human–AI workflow for warehouse slotting recommendations."""

    def __init__(self, num_skus: int = 500, num_locations: int = 50):
        print(
            f"Initializing the human–AI system: {num_skus} SKUs, "
            f"{num_locations} locations"
        )

        self.skus = self._create_skus(num_skus)
        self.locations = self._create_locations(num_locations)
        self.skus_by_id = {sku["id"]: sku for sku in self.skus}
        self.locations_by_id = {
            location["id"]: location for location in self.locations
        }
        self.current_slotting: Dict[int, int] = {}

        self.ai_model: Optional[RandomForestRegressor] = None
        self.scaler: Optional[StandardScaler] = None
        self.feature_names: List[str] = []
        self.model_mape: Optional[float] = None

        self.trust_scores: Dict[int, float] = {}
        self.human_experts = self._create_human_experts()
        self.operational_constraints = self._define_operational_constraints()

        self.decision_history: List[Dict[str, Any]] = []
        self.ai_recommendations: List[Dict[str, Any]] = []
        self.human_decisions: List[Dict[str, Any]] = []
        self.learning_feedback: List[Dict[str, Any]] = []
        self.trust_history: List[Dict[str, Any]] = []

        self._initialize_ai_model()
        self._initialize_slotting()
        print(f"System initialized with {len(self.human_experts)} simulated experts")

    def _create_skus(self, num_skus: int) -> List[Dict[str, Any]]:
        """Create SKUs with synthetic demand and handling attributes."""
        categories = [
            "Electronics", "Clothing", "Food", "Books",
            "Toys", "Sports", "Home", "Beauty",
        ]
        skus = []

        for sku_id in range(1, num_skus + 1):
            category = str(np.random.choice(categories))
            if category == "Food":
                velocity = np.random.lognormal(4.5, 0.8)
            elif category == "Electronics":
                velocity = np.random.lognormal(3.5, 1.2)
            else:
                velocity = np.random.lognormal(3.0, 1.0)

            skus.append(
                {
                    "id": sku_id,
                    "velocity": max(1.0, float(velocity)),
                    "space_req": float(np.random.uniform(0.1, 4.0)),
                    "weight": float(np.random.uniform(0.1, 50.0)),
                    "category": category,
                    "price": float(np.random.uniform(5.0, 500.0)),
                    "seasonality": float(np.random.beta(2, 2)),
                    "fragility": float(np.random.uniform(0.0, 1.0)),
                    "turnover_rate": float(np.random.uniform(0.1, 2.0)),
                    "safety_stock": float(np.random.uniform(0.5, 2.0)),
                }
            )

        return skus

    def _create_locations(self, num_locations: int) -> List[Dict[str, Any]]:
        """Create locations with synthetic capacity and zone attributes."""
        zones = ["Fast-Pick", "Reserve", "Cold-Storage", "High-Security", "Bulk"]
        locations = []

        for location_id in range(1, num_locations + 1):
            zone = str(
                np.random.choice(zones, p=[0.3, 0.4, 0.1, 0.1, 0.1])
            )
            if zone == "Fast-Pick":
                capacity = float(np.random.uniform(8, 20))
                accessibility = float(np.random.uniform(1, 3))
                temperature_control = True
            elif zone == "Reserve":
                capacity = float(np.random.uniform(15, 40))
                accessibility = float(np.random.uniform(4, 8))
                temperature_control = False
            elif zone == "Cold-Storage":
                capacity = float(np.random.uniform(10, 25))
                accessibility = float(np.random.uniform(3, 6))
                temperature_control = True
            else:
                capacity = float(np.random.uniform(20, 50))
                accessibility = float(np.random.uniform(5, 10))
                temperature_control = bool(np.random.choice([True, False]))

            locations.append(
                {
                    "id": location_id,
                    "capacity": capacity,
                    "weight_limit": capacity * 25.0,
                    "accessibility": accessibility,
                    "zone": zone,
                    "temperature_control": temperature_control,
                    "security_level": int(np.random.randint(1, 4)),
                    "maintenance_due": datetime.now()
                    + timedelta(days=int(np.random.randint(0, 30))),
                    "equipment_status": str(
                        np.random.choice(["Good", "Fair", "Poor"], p=[0.7, 0.2, 0.1])
                    ),
                }
            )

        return locations

    def _create_human_experts(self) -> List[Dict[str, Any]]:
        """Create simulated expert profiles and initialize their trust scores."""
        experts = [
            {
                "id": 1,
                "name": "Sarah Chen",
                "role": "Warehouse Manager",
                "experience_years": 15,
                "expertise": [
                    "Fast-Mover Optimization", "Space Utilization", "Seasonal Planning"
                ],
                "decision_style": "Conservative",
                "confidence_level": 0.9,
                "availability": 0.9,
                "bias_tendency": "Safety-First",
            },
            {
                "id": 2,
                "name": "Michael Rodriguez",
                "role": "Operations Analyst",
                "experience_years": 8,
                "expertise": [
                    "Data Analysis", "Process Optimization", "Cost Management"
                ],
                "decision_style": "Analytical",
                "confidence_level": 0.85,
                "availability": 0.95,
                "bias_tendency": "Efficiency-Focused",
            },
            {
                "id": 3,
                "name": "Jennifer Park",
                "role": "Logistics Engineer",
                "experience_years": 12,
                "expertise": [
                    "System Integration", "Automation", "Workflow Design"
                ],
                "decision_style": "Innovative",
                "confidence_level": 0.8,
                "availability": 0.85,
                "bias_tendency": "Technology-Forward",
            },
        ]

        for expert in experts:
            self.trust_scores[expert["id"]] = 0.7

        return experts

    def _define_operational_constraints(self) -> Dict[str, Any]:
        """Store example operating rules; assignment code does not enforce them."""
        return {
            "maintenance_windows": {
                "daily": ["02:00-04:00"],
                "weekly": ["Saturday 03:00-05:00"],
            },
            "staff_availability": {
                "peak_hours": {"start": "09:00", "end": "17:00"},
                "weekend_coverage": 0.6,
                "overtime_budget": 10000,
            },
            "equipment_constraints": {
                "max_concurrent_relocations": 5,
                "relocation_team_size": 3,
                "equipment_downtime": 0.15,
            },
            "business_rules": {
                "min_fast_mover_accessibility": 3.0,
                "max_bulk_distance": 8.0,
                "cold_chain_integrity": 0.95,
                "security_item_isolation": 0.98,
            },
        }

    def _initialize_ai_model(self) -> None:
        """Train a regressor on synthetic scenarios and report held-out MAPE."""
        print("Generating synthetic training scenarios and training the model...")
        training_data = self._generate_training_data()
        X = training_data["features"]
        y = training_data["target"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.ai_model = RandomForestRegressor(n_estimators=100, random_state=42)
        self.ai_model.fit(X_train_scaled, y_train)
        self.feature_names = training_data["feature_names"]

        y_pred = self.ai_model.predict(X_test_scaled)
        self.model_mape = float(mean_absolute_percentage_error(y_test, y_pred))
        print(
            "Synthetic hold-out MAPE: "
            f"{self.model_mape:.3f} (a diagnostic on generated data, not field accuracy)"
        )

    def _generate_training_data(self) -> Dict[str, Any]:
        """Generate random feasible assignments and their constructed score targets."""
        scenarios = []

        for _ in range(1000):
            assignment: Dict[int, int] = {}
            location_usage: Dict[int, Dict[str, float]] = {}
            shuffled_skus = list(self.skus)
            np.random.shuffle(shuffled_skus)

            for sku in shuffled_skus:
                feasible_locations = []
                for location in self.locations:
                    location_id = location["id"]
                    usage = location_usage.setdefault(
                        location_id, {"space": 0.0, "weight": 0.0}
                    )
                    if (
                        usage["space"] + sku["space_req"] <= location["capacity"]
                        and usage["weight"] + sku["weight"]
                        <= location["weight_limit"]
                    ):
                        feasible_locations.append(location)

                if feasible_locations:
                    chosen_location = feasible_locations[
                        int(np.random.randint(len(feasible_locations)))
                    ]
                    location_id = chosen_location["id"]
                    assignment[sku["id"]] = location_id
                    location_usage[location_id]["space"] += sku["space_req"]
                    location_usage[location_id]["weight"] += sku["weight"]

            scenarios.append(
                {
                    "features": self._extract_features(assignment, location_usage),
                    "target": self._calculate_efficiency_score(
                        assignment, location_usage
                    ),
                    "assignment": assignment,
                }
            )

        feature_names = [
            "avg_velocity", "avg_utilization", "util_std", "max_util",
            "min_util", "fast_pick_count", "reserve_count",
            "cold_storage_count", "high_security_count", "bulk_count",
        ]
        return {
            "features": np.asarray([item["features"] for item in scenarios]),
            "target": np.asarray([item["target"] for item in scenarios]),
            "feature_names": feature_names,
            "scenarios": scenarios,
        }

    def _calculate_efficiency_score(
        self,
        assignment: Dict[int, int],
        location_usage: Dict[int, Dict[str, float]],
    ) -> float:
        """Return the mean illustrative score for an assignment."""
        if not assignment:
            return 0.0

        zone_scores = {
            "Fast-Pick": 1.0,
            "Reserve": 0.6,
            "Cold-Storage": 0.8,
            "High-Security": 0.4,
            "Bulk": 0.3,
        }
        total_score = 0.0

        for sku_id, location_id in assignment.items():
            sku = self.skus_by_id[sku_id]
            location = self.locations_by_id[location_id]
            velocity_score = sku["velocity"] / location["accessibility"]
            utilization = (
                location_usage[location_id]["space"] / location["capacity"]
            ) * 100.0
            utilization_bonus = 0.1 if 60.0 <= utilization <= 85.0 else -0.1
            total_score += (
                velocity_score * 0.6
                + zone_scores.get(location["zone"], 0.5) * 0.3
                + utilization_bonus * 0.1
            )

        return total_score / len(assignment)

    def _extract_features(
        self,
        assignment: Dict[int, int],
        location_usage: Dict[int, Dict[str, float]],
    ) -> List[float]:
        """Convert an assignment into the ten model input features."""
        assigned_skus = [self.skus_by_id[sku_id] for sku_id in assignment]
        avg_velocity = (
            float(np.mean([sku["velocity"] for sku in assigned_skus]))
            if assigned_skus
            else 0.0
        )

        utilizations = [
            location_usage[location_id]["space"]
            / self.locations_by_id[location_id]["capacity"]
            for location_id in location_usage
            if location_usage[location_id]["space"] > 0
        ]
        utilization_features = (
            [
                float(np.mean(utilizations)),
                float(np.std(utilizations)),
                float(np.max(utilizations)),
                float(np.min(utilizations)),
            ]
            if utilizations
            else [0.0, 0.0, 0.0, 0.0]
        )

        zone_counts = {
            "Fast-Pick": 0,
            "Reserve": 0,
            "Cold-Storage": 0,
            "High-Security": 0,
            "Bulk": 0,
        }
        for location_id in assignment.values():
            zone_counts[self.locations_by_id[location_id]["zone"]] += 1

        return [avg_velocity] + utilization_features + [
            zone_counts[zone]
            for zone in [
                "Fast-Pick", "Reserve", "Cold-Storage", "High-Security", "Bulk"
            ]
        ]

    def _initialize_slotting(self) -> None:
        """Build a velocity-first feasible assignment as the baseline."""
        print("Creating the baseline slotting assignment...")
        sorted_skus = sorted(
            self.skus, key=lambda item: item["velocity"], reverse=True
        )
        sorted_locations = sorted(
            self.locations, key=lambda item: item["accessibility"]
        )
        location_usage: Dict[int, Dict[str, float]] = {}

        for sku in sorted_skus:
            for location in sorted_locations:
                location_id = location["id"]
                usage = location_usage.setdefault(
                    location_id, {"space": 0.0, "weight": 0.0}
                )
                if (
                    usage["space"] + sku["space_req"] <= location["capacity"]
                    and usage["weight"] + sku["weight"]
                    <= location["weight_limit"]
                ):
                    self.current_slotting[sku["id"]] = location_id
                    usage["space"] += sku["space_req"]
                    usage["weight"] += sku["weight"]
                    break

        print(
            f"Baseline assignment: {len(self.current_slotting)}/{len(self.skus)} "
            "SKUs assigned"
        )

In [ ]:
class HumanAISystem(HumanAISystem):
    """Recommendation, review, feedback, and reporting methods."""

    def generate_ai_recommendations(self) -> List[Dict[str, Any]]:
        """Generate three weighted greedy scenarios and their explanations."""
        print("Generating weighted greedy recommendations...")
        scenarios = [
            ("efficiency_focused", 0.6, 0.3, 0.1),
            ("space_balanced", 0.2, 0.6, 0.2),
            ("balanced", 0.4, 0.4, 0.2),
        ]
        recommendations = []

        for scenario_name, velocity_weight, zone_weight, util_weight in scenarios:
            recommendation = self._generate_scenario_recommendation(
                scenario_name, velocity_weight, zone_weight, util_weight
            )
            recommendation["explanations"] = self._generate_explanations(
                recommendation
            )
            recommendation["confidence_score"] = self._calculate_confidence(
                recommendation
            )
            recommendation["implementation_complexity"] = self._assess_complexity(
                recommendation
            )
            recommendations.append(recommendation)

        self.ai_recommendations = recommendations
        print(f"Generated {len(recommendations)} candidate recommendations")
        return recommendations

    def _generate_scenario_recommendation(
        self,
        scenario_name: str,
        velocity_weight: float,
        zone_weight: float,
        util_weight: float,
    ) -> Dict[str, Any]:
        """Assign SKUs in velocity order using a scenario's weighted score."""
        assignment: Dict[int, int] = {}
        location_usage: Dict[int, Dict[str, float]] = {}
        sorted_skus = sorted(
            self.skus, key=lambda item: item["velocity"], reverse=True
        )
        zone_scores = {
            "Fast-Pick": 1.0,
            "Reserve": 0.6,
            "Cold-Storage": 0.8,
            "High-Security": 0.4,
            "Bulk": 0.3,
        }
        max_velocity = max(sku["velocity"] for sku in self.skus)

        for sku in sorted_skus:
            best_location = None
            best_score = float("-inf")

            for location in self.locations:
                location_id = location["id"]
                usage = location_usage.setdefault(
                    location_id, {"space": 0.0, "weight": 0.0}
                )
                feasible = (
                    usage["space"] + sku["space_req"] <= location["capacity"]
                    and usage["weight"] + sku["weight"]
                    <= location["weight_limit"]
                )
                if not feasible:
                    continue

                velocity_score = (
                    sku["velocity"] / max_velocity
                ) * ((10.0 - location["accessibility"]) / 9.0)
                zone_score = zone_scores[location["zone"]]
                current_utilization = (
                    usage["space"] / location["capacity"]
                ) * 100.0
                if 60.0 <= current_utilization <= 85.0:
                    util_score = 0.1
                elif current_utilization > 85.0:
                    util_score = -0.2
                else:
                    util_score = -0.1

                total_score = (
                    velocity_weight * velocity_score
                    + zone_weight * zone_score
                    + util_weight * util_score
                )
                if total_score > best_score:
                    best_score = total_score
                    best_location = location

            if best_location is not None:
                location_id = best_location["id"]
                assignment[sku["id"]] = location_id
                location_usage[location_id]["space"] += sku["space_req"]
                location_usage[location_id]["weight"] += sku["weight"]

        baseline_usage = self._get_current_usage()
        baseline_score = self._calculate_efficiency_score(
            self.current_slotting, baseline_usage
        )
        recommendation_score = self._calculate_efficiency_score(
            assignment, location_usage
        )
        improvement = (
            (recommendation_score - baseline_score) / abs(baseline_score) * 100.0
            if baseline_score != 0.0
            else 0.0
        )

        features = np.asarray(
            self._extract_features(assignment, location_usage)
        ).reshape(1, -1)
        predicted_score = float(
            self.ai_model.predict(self.scaler.transform(features))[0]
        )

        return {
            "scenario_name": scenario_name,
            "assignment": assignment,
            "expected_improvement": improvement,
            "baseline_score": baseline_score,
            "scenario_score": recommendation_score,
            "predicted_score": predicted_score,
            "velocity_weight": velocity_weight,
            "zone_weight": zone_weight,
            "util_weight": util_weight,
            "location_usage": location_usage,
        }

    def _generate_explanations(
        self, recommendation: Dict[str, Any]
    ) -> List[str]:
        """Describe the recommendation's explicit policy weights."""
        weight_labels = [
            ("velocity_weight", "Fast-moving SKU and accessibility priority"),
            ("zone_weight", "Zone preference"),
            ("util_weight", "Current space-utilization contribution"),
        ]
        ranked_factors = sorted(
            weight_labels,
            key=lambda item: recommendation[item[0]],
            reverse=True,
        )
        explanations = [
            f"{label} (scenario weight: {recommendation[key]:.1f})."
            for key, label in ranked_factors
        ]
        explanations.append(
            "These are rule-based explanations of the greedy policy, not "
            "SHAP or LIME explanations of the random-forest model."
        )
        return explanations

    def _calculate_confidence(self, recommendation: Dict[str, Any]) -> float:
        """Return a heuristic indicator; it is not a calibrated probability."""
        complexity_factor = {
            "efficiency_focused": 0.9,
            "space_balanced": 0.8,
            "balanced": 0.85,
        }.get(recommendation["scenario_name"], 0.8)
        improvement = recommendation["expected_improvement"]
        if improvement > 20.0:
            improvement_factor = 0.95
        elif improvement > 10.0:
            improvement_factor = 0.9
        elif improvement > 5.0:
            improvement_factor = 0.85
        else:
            improvement_factor = 0.8
        return min(0.95, 0.85 * complexity_factor * improvement_factor)

    def _assess_complexity(self, recommendation: Dict[str, Any]) -> str:
        """Classify the fraction of assigned SKUs whose locations would change."""
        changes = sum(
            1
            for sku_id, new_location_id in recommendation["assignment"].items()
            if sku_id in self.current_slotting
            and self.current_slotting[sku_id] != new_location_id
        )
        total_skus = len(recommendation["assignment"])
        change_percentage = changes / total_skus * 100.0 if total_skus else 0.0
        if change_percentage < 10.0:
            return "Low"
        if change_percentage < 25.0:
            return "Medium"
        return "High"

    def _get_current_usage(self) -> Dict[int, Dict[str, float]]:
        """Calculate space and weight totals for the baseline assignment."""
        usage: Dict[int, Dict[str, float]] = {}
        for sku_id, location_id in self.current_slotting.items():
            sku = self.skus_by_id[sku_id]
            totals = usage.setdefault(
                location_id, {"space": 0.0, "weight": 0.0}
            )
            totals["space"] += sku["space_req"]
            totals["weight"] += sku["weight"]
        return usage

    def human_review_process(
        self, recommendations: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Apply the explicit simulated review rules and record each decision."""
        print("\n=== SIMULATED HUMAN REVIEW ===")
        reviewed_recommendations = []

        for index, recommendation in enumerate(recommendations, start=1):
            expert = self._select_expert_for_review(recommendation)
            if expert is None:
                print(f"No available expert for recommendation {index}")
                continue

            print(
                f"\nRecommendation {index}: {recommendation['scenario_name']} | "
                f"Expert: {expert['name']} | "
                f"Heuristic confidence: {recommendation['confidence_score']:.2f} | "
                f"Score change: {recommendation['expected_improvement']:.1f}%"
            )
            for explanation in recommendation["explanations"]:
                print(f"  - {explanation}")

            decision = self._human_expert_analysis(expert, recommendation)
            record = {
                "recommendation": recommendation,
                "expert": expert,
                "decision": decision,
                "timestamp": datetime.now(),
            }
            reviewed_recommendations.append(record)
            self.decision_history.append(record)
            self._update_trust_scores(expert, recommendation, decision)

        self.ai_recommendations = recommendations
        self.human_decisions = reviewed_recommendations
        accepted_or_modified = sum(
            record["decision"]["action"] in ["accept", "modify"]
            for record in reviewed_recommendations
        )
        acceptance_rate = (
            accepted_or_modified / len(reviewed_recommendations) * 100.0
            if reviewed_recommendations
            else 0.0
        )
        modified = sum(
            record["decision"]["action"] == "modify"
            for record in reviewed_recommendations
        )
        rejected = sum(
            record["decision"]["action"] == "reject"
            for record in reviewed_recommendations
        )
        print("\nReview summary:")
        print(f"  Reviewed: {len(reviewed_recommendations)}")
        print(f"  Accepted or modified: {accepted_or_modified} ({acceptance_rate:.1f}%)")
        print(f"  Modified: {modified}")
        print(f"  Rejected: {rejected}")
        return reviewed_recommendations

    def _select_expert_for_review(
        self, recommendation: Dict[str, Any]
    ) -> Optional[Dict[str, Any]]:
        """Select an available expert whose stated expertise is relevant."""
        available_experts = [
            expert
            for expert in self.human_experts
            if random.random() < expert["availability"]
        ]
        relevant_experts = []
        for expert in available_experts:
            expertise = expert["expertise"]
            if (
                "Fast-Mover Optimization" in expertise
                and recommendation["velocity_weight"] > 0.5
            ):
                relevant_experts.append(expert)
            elif (
                "Space Utilization" in expertise
                and recommendation["util_weight"] > 0.5
            ):
                relevant_experts.append(expert)
            elif "Process Optimization" in expertise:
                relevant_experts.append(expert)

        candidates = relevant_experts or available_experts
        return (
            max(candidates, key=lambda expert: expert["confidence_level"])
            if candidates
            else None
        )

    def _human_expert_analysis(
        self, expert: Dict[str, Any], recommendation: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Simulate a review decision using stated thresholds and expert rules."""
        decision_score = 0.0
        reasoning = []
        improvement = recommendation["expected_improvement"]
        confidence = recommendation["confidence_score"]

        if improvement < 5.0:
            decision_score -= 0.1
            reasoning.append("Small or negative illustrative score change")
        elif improvement > 15.0:
            decision_score += 0.1
            reasoning.append("Large illustrative score change")

        if confidence > 0.8:
            decision_score += 0.2
            reasoning.append("High heuristic confidence indicator")
        elif confidence < 0.6:
            decision_score -= 0.2
            reasoning.append("Low heuristic confidence indicator")

        if (
            expert["bias_tendency"] == "Safety-First"
            and recommendation["implementation_complexity"] == "High"
        ):
            decision_score -= 0.2
            reasoning.append("High change level under the safety-first rule")
        elif (
            expert["bias_tendency"] == "Technology-Forward"
            and recommendation["implementation_complexity"] == "High"
        ):
            decision_score += 0.1
            reasoning.append("High change level under the technology-forward rule")

        if decision_score >= 0.2:
            if expert["confidence_level"] > 0.8 and improvement > 10.0:
                action = "accept"
                reasoning.append("Simulated policy selects accept")
            else:
                action = "modify"
                reasoning.append("Simulated policy selects modify")
        elif decision_score >= 0.0:
            action = "modify"
            reasoning.append("Simulated policy selects modify")
        else:
            action = "reject"
            reasoning.append("Simulated policy selects reject")

        return {
            "action": action,
            "reasoning": (
                f"{expert['name']} ({expert['role']}, "
                f"{expert['decision_style']} style): "
                + "; ".join(reasoning)
            ),
            "decision_score": decision_score,
        }

    def _update_trust_scores(
        self,
        expert: Dict[str, Any],
        recommendation: Dict[str, Any],
        decision: Dict[str, Any],
    ) -> None:
        """Update the simulated trust record; this does not train the model."""
        expert_id = expert["id"]
        old_trust = self.trust_scores[expert_id]
        action = decision["action"]

        if action == "accept":
            increment = (
                0.05
                if recommendation["confidence_score"] > 0.8
                and recommendation["expected_improvement"] > 10.0
                else 0.02
            )
            new_trust = min(0.95, old_trust + increment)
        elif action == "modify":
            new_trust = min(0.95, old_trust + 0.01)
        else:
            decrement = (
                0.05
                if recommendation["confidence_score"] < 0.6
                or recommendation["expected_improvement"] < 5.0
                else 0.02
            )
            new_trust = max(0.5, old_trust - decrement)

        self.trust_scores[expert_id] = new_trust
        feedback = {
            "timestamp": datetime.now(),
            "expert_id": expert_id,
            "action": action,
            "old_trust": old_trust,
            "new_trust": new_trust,
            "confidence_indicator": recommendation["confidence_score"],
        }
        self.learning_feedback.append(feedback)
        self.trust_history.append(
            {"expert_id": expert_id, "trust_score": new_trust}
        )

    def generate_performance_dashboard(self) -> None:
        """Plot recorded trust, review, contribution, and score-change metrics."""
        print("\n=== HUMAN–AI PROCESS DASHBOARD ===")
        fig, axes = plt.subplots(2, 3, figsize=(18, 11))

        if self.trust_history:
            for expert in self.human_experts:
                scores = [
                    item["trust_score"]
                    for item in self.trust_history
                    if item["expert_id"] == expert["id"]
                ]
                if scores:
                    axes[0, 0].plot(
                        range(1, len(scores) + 1),
                        scores,
                        marker="o",
                        label=expert["name"],
                    )
            axes[0, 0].set_title("Recorded Trust Scores")
            axes[0, 0].set_xlabel("Recorded updates per expert")
            axes[0, 0].set_ylabel("Trust score (simulation units)")
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)

        if self.human_decisions:
            confidences = [
                item["recommendation"]["confidence_score"]
                for item in self.human_decisions
            ]
            axes[0, 1].plot(
                range(1, len(confidences) + 1),
                confidences,
                marker="s",
                color="green",
            )
            axes[0, 1].set_title("Heuristic Confidence Indicators")
            axes[0, 1].set_xlabel("Reviewed recommendation")
            axes[0, 1].set_ylabel("Indicator (not a probability)")
            axes[0, 1].grid(True, alpha=0.3)

            actions = [item["decision"]["action"] for item in self.human_decisions]
            action_names = ["accept", "modify", "reject"]
            action_counts = [actions.count(name) for name in action_names]
            axes[0, 2].pie(
                action_counts,
                labels=action_names,
                autopct="%1.1f%%",
                startangle=90,
                colors=["lightgreen", "lightblue", "lightcoral"],
            )
            axes[0, 2].set_title("Simulated Review Decisions")
            axes[0, 2].axis("equal")

            expert_counts = {
                expert["id"]: sum(
                    item["expert"]["id"] == expert["id"]
                    for item in self.human_decisions
                )
                for expert in self.human_experts
            }
            axes[1, 0].bar(
                list(expert_counts),
                list(expert_counts.values()),
                color="skyblue",
                alpha=0.8,
            )
            axes[1, 0].set_title("Expert Review Assignments")
            axes[1, 0].set_xlabel("Expert ID")
            axes[1, 0].set_ylabel("Number of decisions")
            axes[1, 0].grid(True, axis="y", alpha=0.3)

            score_changes = [
                item["recommendation"]["expected_improvement"]
                for item in self.human_decisions
            ]
            axes[1, 1].hist(
                score_changes,
                bins=min(10, max(1, len(score_changes))),
                color="lightgreen",
                alpha=0.8,
                edgecolor="black",
            )
            axes[1, 1].set_title("Illustrative Score Changes")
            axes[1, 1].set_xlabel("Change from baseline score (%)")
            axes[1, 1].set_ylabel("Recommendation count")
            axes[1, 1].grid(True, axis="y", alpha=0.3)

        if self.learning_feedback:
            batch_size = 3
            acceptance_rates = []
            for start in range(0, len(self.learning_feedback), batch_size):
                batch = self.learning_feedback[start : start + batch_size]
                accepted = sum(
                    item["action"] in ["accept", "modify"] for item in batch
                )
                acceptance_rates.append(accepted / len(batch) * 100.0)
            axes[1, 2].plot(
                range(1, len(acceptance_rates) + 1),
                acceptance_rates,
                marker="^",
                color="purple",
            )
            axes[1, 2].set_title("Review Rate by Decision Batch")
            axes[1, 2].set_xlabel("Batch of up to three reviews")
            axes[1, 2].set_ylabel("Accepted or modified (%)")
            axes[1, 2].set_ylim(0, 100)
            axes[1, 2].grid(True, alpha=0.3)

        for axis in axes.flat:
            if not axis.has_data():
                axis.set_title("No recorded data")
                axis.set_axis_off()

        fig.tight_layout()
        plt.show()

    def simulate_learning_cycle(self, num_cycles: int = 6) -> Dict[str, Any]:
        """Repeat recommendation and review; feedback is recorded, not retrained."""
        if num_cycles <= 0:
            raise ValueError("num_cycles must be a positive integer")

        print(f"\n=== SIMULATING {num_cycles} REVIEW CYCLES ===")
        cycles_data = []

        for cycle in range(num_cycles):
            print(f"\n--- Review cycle {cycle + 1}/{num_cycles} ---")
            recommendations = self.generate_ai_recommendations()
            reviewed = self.human_review_process(recommendations)
            acceptance_rate = (
                sum(
                    item["decision"]["action"] in ["accept", "modify"]
                    for item in reviewed
                )
                / len(reviewed)
                * 100.0
                if reviewed
                else 0.0
            )
            avg_trust = (
                float(np.mean(list(self.trust_scores.values())))
                if self.trust_scores
                else 0.0
            )
            avg_confidence = (
                float(
                    np.mean(
                        [item["confidence_score"] for item in recommendations]
                    )
                )
                if recommendations
                else 0.0
            )
            avg_score_change = (
                float(
                    np.mean(
                        [
                            item["recommendation"]["expected_improvement"]
                            for item in reviewed
                        ]
                    )
                )
                if reviewed
                else 0.0
            )
            cycles_data.append(
                {
                    "cycle": cycle + 1,
                    "acceptance_rate": acceptance_rate,
                    "avg_trust": avg_trust,
                    "avg_confidence": avg_confidence,
                    "avg_score_change": avg_score_change,
                }
            )

        results = {
            "cycles_data": cycles_data,
            "final_acceptance_rate": float(
                np.mean([item["acceptance_rate"] for item in cycles_data])
            ),
            "final_trust_score": float(
                np.mean([item["avg_trust"] for item in cycles_data])
            ),
            "final_confidence": float(
                np.mean([item["avg_confidence"] for item in cycles_data])
            ),
            "final_score_change": float(
                np.mean([item["avg_score_change"] for item in cycles_data])
            ),
        }

        print("\nCycle averages (simulation outputs):")
        print(f"  Accepted or modified: {results['final_acceptance_rate']:.1f}%")
        print(f"  Mean recorded trust: {results['final_trust_score']:.2f}")
        print(f"  Mean heuristic confidence: {results['final_confidence']:.2f}")
        print(f"  Mean illustrative score change: {results['final_score_change']:.1f}%")
        self.generate_performance_dashboard()
        return results

In [ ]:
# Initialize the example at the scale specified for this simulation.
human_ai_system = HumanAISystem(num_skus=200, num_locations=30)

# Run repeated recommendation and simulated review cycles.
learning_results = human_ai_system.simulate_learning_cycle(num_cycles=6)

# Generate and review one additional set of recommendations for inspection.
ai_recommendations = human_ai_system.generate_ai_recommendations()
reviewed_recommendations = human_ai_system.human_review_process(ai_recommendations)

print("\n=== FINAL SIMULATION SUMMARY ===")
print(f"Mean accepted-or-modified rate: {learning_results['final_acceptance_rate']:.1f}%")
print(f"Mean recorded trust score: {learning_results['final_trust_score']:.2f}")
print(f"Mean heuristic confidence indicator: {learning_results['final_confidence']:.2f}")
print(f"Mean illustrative score change: {learning_results['final_score_change']:.1f}%")
print(f"Synthetic hold-out MAPE: {human_ai_system.model_mape:.3f}")

print("\nExpert trust scores after the additional review:")
for expert in human_ai_system.human_experts:
    score = human_ai_system.trust_scores[expert["id"]]
    print(f"  {expert['name']}: {score:.2f}")

# Check that each recommendation respects the two modeled resource limits.
for recommendation in ai_recommendations:
    for location_id, usage in recommendation["location_usage"].items():
        location = human_ai_system.locations_by_id[location_id]
        assert usage["space"] <= location["capacity"] + 1e-9
        assert usage["weight"] <= location["weight_limit"] + 1e-9

print("\nFeasibility checks passed for the modeled space and weight limits.")

## Results analysis

The preceding cells produce the results for this run. Interpret them against the assumptions and limitations stated above rather than as fixed targets.

### Reading the metrics

- **Synthetic hold-out MAPE** measures the random forest's percentage prediction error on a held-out portion of the generated scenarios. Because both training and test targets come from the same synthetic score construction, this metric does not measure accuracy on real warehouse outcomes.
- **Illustrative score change** compares the constructed assignment score with the baseline score. It is not a percentage reduction in picking time or a measured productivity improvement.
- **Heuristic confidence** is a rule-based indicator assembled from scenario and score-change thresholds. It is not a probability and has not been calibrated against observed recommendation outcomes.
- **Accepted-or-modified rate** is the share of reviewed candidates assigned either simulated action. It is not a measure of quality or a forecast of manager behavior.
- **Recorded trust score** follows the update rules in `_update_trust_scores`. It summarizes the simulation's bookkeeping and does not represent validated human trust.

### What the workflow demonstrates

The sequence makes a collaborative decision process explicit: a system proposes candidates, exposes the policy weights behind them, and places a simulated review step between recommendations and operational action. The separate functions for candidate generation, explanation, review, and trust recording make the responsibilities inspectable.

The repeated cycles do not retrain the random forest, change the scenario weights, or incorporate expert feedback into later assignments. Therefore, changes in review and trust metrics across cycles arise from the stated decision rules and simulated expert availability, not from model learning. A production human-in-the-loop system would need a defined feedback target, reliable outcome data, governance for expert input, and evaluation that distinguishes model performance from human-review behavior.

### Scope of conclusions

This example illustrates how transparent heuristics, a learned score estimator, and human oversight can be organized in one workflow. It does not show that the combined process outperforms a human-only or AI-only process, nor does it validate the listed operational constraints as implemented assignment rules. Any real deployment would require additional warehouse-specific feasibility constraints, measured costs and outcomes, and expert validation.